# Retrain Joint mBERT (en/es/hi/te) — seeds 42, 123, 7

**Why this exists:** the `joint_mbert` checkpoints for `en_es_hi_te` (seeds 42/123/7) lost their fine-tuned BERT encoder weights at some point — only `task_heads.pt` (the 3 small linear heads) survived on Drive, everywhere checked. Running inference with those alone gives chance-level results (verified: 0.51 cls accuracy on English, the model's own training language). This notebook retrains all 3 seeds from scratch via `training/Train_Join.py` and, critically, **persists the full encoder** (`model.safetensors`) to Drive this time — that's exactly what got dropped last time.

T4 is fine (matches the original 9-system taxonomy's runtime, mBERT-base is small). Each seed should take well under an hour.

**After this notebook finishes:** come back to the main session — the N2 French/German zero-shot eval and the historical 5-language sanity check both resume from these checkpoints.

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEEDS    = [42, 123, 7]
DRIVE_ROOT = '/content/drive/MyDrive/Idiomator_Research'
print('repo:', REPO, '| seeds:', SEEDS)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd
import os, subprocess, sys
from pathlib import Path
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert Path('training/Train_Join.py').exists(), 'Train_Join.py missing — wrong repo/branch?'

In [ ]:
# 3. Install deps + confirm GPU (T4 is fine)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive, symlink models/ so Train_Join.py's output_dir writes straight to Drive
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_ROOT, 'models').mkdir(parents=True, exist_ok=True)
local_models = Path(REPO, 'models')
if local_models.is_symlink() or local_models.exists():
    if not local_models.is_symlink():
        subprocess.run(['mv', str(local_models), str(local_models) + '_bak'], check=True)
if not local_models.exists():
    os.symlink(f'{DRIVE_ROOT}/models', str(local_models))
print('models/ ->', os.path.realpath(local_models))

In [ ]:
# 5. Smoke test (~1-2 min): 1 epoch, English only, throwaway dir — catches path/CLI bugs before real spend
!python training/Train_Join.py \
    --langs English --output_dir /tmp/_joint_smoke --epochs 1 --batch_size 8 --seed 42

In [ ]:
# 6. FULL RUNS — 3 seeds, en/es/hi/te, output straight to Drive-symlinked models/.
#    Re-run this cell after any disconnect; each seed's output_dir is separate so finished ones are untouched.
OUT_DIRS = {42: 'models/en_es_hi_te/joint_mbert', 123: 'models/main_s123/joint_mbert', 7: 'models/main_s7/joint_mbert'}
for seed in SEEDS:
    out_dir = OUT_DIRS[seed]
    print(f'\n=== seed {seed} -> {out_dir} ===')
    !python training/Train_Join.py --langs English Spanish Hindi Telugu \
        --test_langs English Spanish Hindi Telugu Indonesian \
        --output_dir {out_dir} --seed {seed}

In [ ]:
# 7. Persistence check — confirms the encoder actually landed on Drive this time (the thing that got lost before)
for seed, out_dir in OUT_DIRS.items():
    best = Path(out_dir) / 'best_model'
    has_encoder = (best / 'model.safetensors').exists() or (best / 'pytorch_model.bin').exists()
    has_heads   = (best / 'task_heads.pt').exists()
    flag = 'OK' if has_encoder and has_heads else 'MISSING FILES — do not treat as done'
    print(f'seed {seed:<4} encoder={has_encoder} heads={has_heads}  [{flag}]')
print('\nIf all say OK: come back to the main session, checkpoints are ready to pull.')